In [ ]:
import numpy as np
from astropy.io import fits as fits
import healpy as hp
from astropy.wcs import WCS
from reproject import reproject_from_healpix, reproject_to_healpix
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import gc
import matplotlib as mpl
from astropy.coordinates import SkyCoord, Galactic

In [ ]:
c_hdu_FD  = fits.open('/srv/data/chime/chime_FD_May2024_400_729/FDF_clean_tot.hpx512.fits')
d_hdu_FD  = fits.open('/srv/data/dragons/chime_compare_400_729/faraday_synthesis/FDF_clean_tot.hpx128.fits')
cd_hdu_FD = fits.open('/srv/data/dragons/chime_compare_400_729/faraday_synthesis/FDF_clean_DRAGONS_CHIME.fits')

c_FDF  = c_hdu_FD[0].data
d_FDF  = d_hdu_FD[0].data
cd_FDF = cd_hdu_FD[0].data

In [ ]:
c_FDF.shape

In [ ]:
c_hdu_Q = fits.open('/srv/data/chime/chime_IQUV_Mar2024_400_729/Q_400_729_Mar2024_new_gal_fix.fits')
c_hdu_U = fits.open('/srv/data/chime/chime_IQUV_Mar2024_400_729/U_400_729_Mar2024_new_gal_fix.fits')

c_Q = c_hdu_Q[0].data
c_U = c_hdu_U[0].data

In [ ]:
c_U[200].shape

In [ ]:
def patch_header(lmid, bmid, dtheta):

    dl = dtheta/np.cos(bmid*np.pi/180)
    db = dtheta
    
    l1 = lmid - dl
    l2 = lmid + dl
    b1 = bmid - db
    b2 = bmid + db

    # Make a header and WCS:
    hdu = fits.PrimaryHDU()
    hdr = hdu.header
    hdr['NAXIS'] = 2
    hdr['NAXIS1'] = 1440
    hdr['NAXIS2'] = 720
    hdr['CRVAL1'] = lmid 
    hdr['CDELT1'] = -0.25
    hdr['CRPIX1'] = 720.25
    hdr['CRVAL2'] = bmid
    hdr['CDELT2'] = 0.25
    hdr['CRPIX2'] = 360.25
    hdr['COORD'] = 'galactic'
    hdr['CTYPE1']  = 'GLON-SIN' # To make it have Mollweide projection
    hdr['CTYPE2']  = 'GLAT-SIN'
    
    return hdr, l1, l2, b1, b2

In [ ]:
def make_patch_map(map, hpx_pc, hdr_patch, hdr=None):

    if hpx_pc == 'hpx':
        map_out, _ = reproject_from_healpix((map,'galactic'), hdr_patch, nested=False)
    else:
        hdr2D = WCS(hdr).dropaxis(2).to_header()
        map_hpx, _ = reproject_to_healpix((map, hdr2D), 'galactic', nside=512)
        map_out, _ = reproject_from_healpix((map_hpx,'galactic'), hdr_patch, nested=False)

    return map_out

In [ ]:
lmid = 160
bmid = -10

hdr_patch, l1, l2, b1, b2 = patch_header(lmid, bmid, 20)

In [ ]:
maps = []

fidx = 300

maps.append(make_patch_map(c_Q[fidx],  'pc', hdr_patch, hdr=c_hdu_Q[0].header))
maps.append(make_patch_map(c_U[fidx],  'pc', hdr_patch, hdr=c_hdu_U[0].header))
maps.append(make_patch_map(c_FDF[400], 'hpx', hdr_patch))


In [ ]:
def plot_patch(maps, hdr_patch, l1,l2,b1,b2,lmid,bmid, fs=26, RMmax=10):

    fig = plt.figure(figsize=(24, 7))
    cmap = mpl.colormaps.get_cmap('RdBu_r')
    cmap.set_bad(color='grey')

    wcs = WCS(hdr_patch)

    ax1 = fig.add_subplot(131, projection=wcs)
    ax2 = fig.add_subplot(132, projection=wcs)
    ax3 = fig.add_subplot(133, projection=wcs)

    axs = [ax1, ax2, ax3]
    ims = []
    vmins = [-2, -2, 0]
    vmaxs = [ 2,  2, 5]

    plt.subplots_adjust(left=0.05, bottom=0.15, right=0.85, top=0.99, wspace=0.3)

    for i in range(0,len(maps)):
        im = axs[i].imshow(maps[i], origin='lower', cmap=cmap, vmin=vmins[i], vmax=vmaxs[i])
        ims.append(im)

    cx = SkyCoord([l2,l1], [bmid,bmid], frame=Galactic, unit="deg")
    cy = SkyCoord([lmid,lmid], [b1,b2], frame=Galactic, unit="deg")

    for i in range(0,3):
        axs[i].coords.grid()
        axs[i].set_xlim(wcs.world_to_pixel(cx)[0])
        axs[i].set_ylim(wcs.world_to_pixel(cy)[1])
        axs[i].set_xlabel(' ')
        axs[i].set_ylabel(' ')

        for spine in axs[i].spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)
        axs[i].tick_params(axis='both', labelsize=fs, which='both', width=2, length=6)

        #cat_coords_lofar = SkyCoord(cat['l'], cat['b'], frame=Galactic, unit="deg")
    
        #ax1.scatter(wcs.world_to_pixel(cat_coords_lofar)[0], wcs.world_to_pixel(cat_coords_lofar)[1], 
        #           s=50, color='k')

        axs[i].set_ylabel('Galactic Latitude',fontsize=fs)
        axs[i].set_xlabel('Galactic Longitude',fontsize=fs)

    return

In [ ]:
plot_patch(maps, hdr_patch, l1,l2,b1,b2,lmid,bmid)
